# Chapter 13 — Deploying scikit-learn Models in Production

**Book:** scikit-learn Cookbook, Third Edition — John Sukup (Packt Publishing, 2025)

This notebook follows the chapter sequence and demonstrates the core methods through executable examples. Numerical results appear immediately below the code, followed by a discussion of how to read them. The code examples are self-contained educational reproductions and extensions, not a word-for-word transcription of the text.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
np.set_printoptions(precision=3, suppress=True)
RANDOM_STATE=42

## 1. 1. Overview of model deployment

Deployment means making a trained model available to make predictions outside the notebook. The preprocessing steps used during training must travel with the model.

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X,y=load_iris(return_X_y=True)
A,B,c,d=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)).fit(A,c)
print('Validation accuracy:',round(model.score(B,d),3))

Validation accuracy: 0.921


**Interpretation.** The fitted pipeline acts as a single reusable prediction component. Saving only the classifier without its preprocessing could lead to inconsistent inputs after deployment.

## 2. 2. Serialization with joblib

Serialization saves a fitted Python object so that it can be restored later. Only load model files from trusted sources, because pickle-based formats can execute code.

In [3]:
import joblib
from pathlib import Path
path=Path('iris_pipeline.joblib')
joblib.dump(model,path)
loaded=joblib.load(path)
print('Saved file:',path.name)
print('Identical predictions:',(model.predict(B)==loaded.predict(B)).all())
path.unlink()

Saved file: iris_pipeline.joblib
Identical predictions: True


**Interpretation.** Identical predictions demonstrate that the reloaded pipeline preserves its fitted parameters. The example removes the temporary file so the notebook remains tidy.

## 3. 3. Input schema and prediction checks

Production inference should validate the number and type of incoming features rather than assuming the caller sent a correct array.

In [4]:
import numpy as np
def predict_checked(four_features):
    row=np.asarray(four_features,dtype=float).reshape(1,-1)
    if row.shape[1]!=4 or not np.isfinite(row).all():
        raise ValueError('Expected four finite numeric features')
    return {'class':int(model.predict(row)[0]),'probabilities':model.predict_proba(row)[0].round(3).tolist()}
print(predict_checked([5.1,3.5,1.4,.2]))

{'class': 0, 'probabilities': [0.981, 0.019, 0.0]}


**Interpretation.** Input validation prevents malformed values from quietly passing into the estimator. The returned class and probability vector provide a simple application-facing prediction response.

## 4. 4. Batch scoring and throughput

For tabular scikit-learn models, predicting a batch is often more efficient than repeatedly calling prediction for individual observations.

In [5]:
import time
batch=np.tile(B[:10],(100,1))
t0=time.perf_counter()
result=model.predict(batch)
duration=time.perf_counter()-t0
print('Batch rows:',len(result))
print('Elapsed seconds:',round(duration,5))

Batch rows: 1000
Elapsed seconds: 0.00037


**Interpretation.** The elapsed time is only an illustrative local measurement; hardware and deployment environment affect throughput. Batch inference reduces per-call overhead when many rows are available.

## 5. 5. Monitoring prediction quality

Once true labels arrive, deployed models can be evaluated again. Decreasing accuracy over time can be a warning sign that data or relationships have changed.

In [6]:
from sklearn.metrics import accuracy_score
pred=model.predict(B)
chunks=np.array_split(np.arange(len(d)),3)
for i,idx in enumerate(chunks,1):
    print('Period',i,'accuracy',round(accuracy_score(d[idx],pred[idx]),3),'examples',len(idx))

Period 1 accuracy 0.846 examples 13
Period 2 accuracy 1.0 examples 13
Period 3 accuracy 0.917 examples 12


**Interpretation.** Each simulated period uses a slice of the held-out data; the numbers are not evidence of real drift. A real monitoring system would store timestamps, inputs, predictions, and delayed actual outcomes.

## 6. 6. Deployment quality gate

A simple quality gate can require a model to reach a minimum evaluation score before it is considered ready for delivery.

In [7]:
from sklearn.metrics import f1_score
score=f1_score(d,model.predict(B),average='macro')
minimum=.85
print('Macro F1:',round(score,3))
print('Threshold:',minimum)
print('Deployment candidate:', 'PASS' if score>=minimum else 'REVIEW')

Macro F1: 0.923
Threshold: 0.85
Deployment candidate: PASS


**Interpretation.** The threshold is an example, not a universal industry standard. Real release gates should also test latency, data validation, reproducibility, security, and expected business impact.

## 7. Training an artifact for deployment

A production artifact must package all necessary preprocessing with the fitted estimator.

In [8]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
D=load_iris();Xa,Xb,ya,yb=train_test_split(D.data,D.target,test_size=.2,random_state=42,stratify=D.target)
pipe=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(Xa,ya)
print('Holdout accuracy:',round(pipe.score(Xb,yb),3))

Holdout accuracy: 0.933


**Interpretation.** A pipeline avoids depending on separately recreated scaling calculations. The holdout score is a useful smoke test but not an ongoing production guarantee.

## 8. Serializing a fitted pipeline

joblib can save estimators efficiently. Only load serialized files from trusted sources, because deserialization can execute code.

In [9]:
import joblib,tempfile,os
path=os.path.join(tempfile.gettempdir(),'cookbook_deployment_model.joblib')
joblib.dump(pipe,path)
print('Saved bytes:',os.path.getsize(path))

Saved bytes: 1569


**Interpretation.** Saving a pipeline preserves learned preprocessing statistics and model parameters. The environment and package versions still matter when restoring it.

## 9. Reloading the model and verifying predictions

Restored predictions should match those made before serialization on a fixed test batch.

In [10]:
restored=joblib.load(path)
a=pipe.predict(Xb[:8]);b=restored.predict(Xb[:8]);print('Predictions match:',np.array_equal(a,b));print('Predictions:',b)

Predictions match: True
Predictions: [0 2 1 1 0 1 0 0]


**Interpretation.** Matching predictions provide a simple consistency check that the file contains the expected fitted object. Passing this test does not establish security or backward compatibility.

## 10. Inspecting model metadata

Deployment artifacts benefit from recording the training library version, feature names and timestamp.

In [11]:
import sklearn,sys,datetime
meta={'python':sys.version.split()[0],'sklearn':sklearn.__version__,'features':D.feature_names,'recorded_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
print(meta)

{'python': '3.13.5', 'sklearn': '1.8.0', 'features': ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'], 'recorded_utc': '2026-10-25T14:19:11.152490+00:00'}


**Interpretation.** Metadata helps diagnose differences between runtime environments. It should be stored alongside, rather than substituted for, an immutable model build identifier.

## 11. Validating the incoming feature schema

Model prediction expects features in the same order and units as during training.

In [12]:
def validate_array(values,feature_count):
 arr=np.asarray(values,dtype=float)
 if arr.ndim!=2 or arr.shape[1]!=feature_count:raise ValueError('Wrong feature dimensions')
 if not np.isfinite(arr).all():raise ValueError('Missing or infinite inputs')
 return arr
batch=validate_array(Xb[:3],D.data.shape[1]);print('Validated batch:',batch.shape)

Validated batch: (3, 4)


**Interpretation.** Checking feature count and numeric finiteness catches common input mistakes before inference. Production code should additionally check allowed ranges and schema versions.

## 12. Batch inference

Vectorized batch prediction reduces overhead compared with repeatedly constructing individual prediction requests.

In [13]:
batch=validate_array(Xb[:12],4)
print('Batch labels:',restored.predict(batch).tolist());print('Probabilities for first row:',restored.predict_proba(batch[:1]).round(3))

Batch labels: [0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2]
Probabilities for first row: [[0.979 0.021 0.   ]]


**Interpretation.** Probability outputs provide additional information for downstream thresholds or review workflows. They should not be assumed to be perfectly calibrated.

## 13. A simple inference function

A clearly defined prediction function isolates model loading and validation from calling code.

In [14]:
def predict_species(records,model):
 features=validate_array(records,4)
 return [{'label':D.target_names[int(i)],'confidence':float(max(p))} for i,p in zip(model.predict(features),model.predict_proba(features))]
print(predict_species(Xb[:2],restored))

[{'label': np.str_('setosa'), 'confidence': 0.9788180050826797}, {'label': np.str_('virginica'), 'confidence': 0.6269814626174681}]


**Interpretation.** A small inference interface makes it easier to test contract behavior. In a real service, inputs and outputs would be validated against an API specification.

## 14. Training-validation deployment gate

One deployment check is to require that the model score surpass a minimum threshold on held-out data.

In [15]:
from sklearn.metrics import accuracy_score
score=accuracy_score(yb,restored.predict(Xb));minimum=.8
print('Holdout score:',round(score,3),'deployment approved:',bool(score>=minimum))

Holdout score: 0.933 deployment approved: True


**Interpretation.** The threshold creates a clear pass-or-fail rule. It should be determined by business risk and historical performance rather than chosen arbitrarily after observing results.

## 15. Monitoring class frequency shift

Incoming label or prediction distributions can drift even when the interface continues to run.

In [16]:
from collections import Counter
train_freq=np.bincount(pipe.predict(Xa),minlength=3)/len(Xa)
test_freq=np.bincount(pipe.predict(Xb),minlength=3)/len(Xb)
print('Training predicted frequency:',train_freq.round(3));print('Holdout predicted frequency:',test_freq.round(3))

Training predicted frequency: [0.333 0.342 0.325]
Holdout predicted frequency: [0.333 0.333 0.333]


**Interpretation.** Distribution changes may be early warnings of data drift but can also arise from small samples. Comparing prediction frequencies alone does not prove a model has become inaccurate.

## 16. Detecting feature drift through mean shift

Feature-level reference statistics make it possible to check how live records differ from training inputs.

In [17]:
reference_mean=Xa.mean(axis=0);incoming=np.array(Xb,copy=True);incoming[:,0]+=0.5
shift=incoming.mean(axis=0)-reference_mean
print('Per-feature mean shift:',shift.round(3))

Per-feature mean shift: [ 0.508  0.045 -0.06  -0.028]


**Interpretation.** Mean shifts flag possible changes in population or instrumentation. A threshold should reflect reference variability and expected seasonality.

## 17. Monitoring error after labels arrive

Actual performance can be measured only after labels or verified outcomes become available.

In [18]:
from sklearn.metrics import classification_report
print(classification_report(yb,restored.predict(Xb),target_names=D.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



**Interpretation.** Precision, recall and support reveal class-level strengths and weaknesses. Monitoring these over time is more useful than relying only on training scores.

## 18. Retraining when data changes

Periodic retraining should be tied to sufficient new labeled data and a repeatable validation process.

In [19]:
updated=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(D.data,D.target)
print('Current evaluation baseline:',round(pipe.score(Xb,yb),3));print('Updated pipeline fitted on:',len(D.data),'records')

Current evaluation baseline: 0.933
Updated pipeline fitted on: 150 records


**Interpretation.** The updated model is trained using all available data and must not be evaluated on the same observations as if they were unseen. New held-out data or future outcomes are required before claiming improvement.

## 19. Model lifecycle checks

A minimal release checklist can validate estimator readiness, expected feature count and reproducible predictions.

In [20]:
from sklearn.utils.validation import check_is_fitted
check_is_fitted(restored.named_steps['logisticregression'])
assert restored.n_features_in_==4
assert np.array_equal(restored.predict(Xb[:4]),pipe.predict(Xb[:4]))
print('Model release smoke checks passed')

Model release smoke checks passed


**Interpretation.** Smoke tests verify basic readiness and a stable prediction contract. Integration tests, authentication, logging, monitoring and rollback procedures remain additional production responsibilities.

## 20. Comparing simple deployment versions

Keeping candidate model results separately helps avoid silently replacing an existing model without evaluation.

In [21]:
from sklearn.tree import DecisionTreeClassifier
candidate=make_pipeline(StandardScaler(),DecisionTreeClassifier(max_depth=3,random_state=42)).fit(Xa,ya)
print('Baseline holdout:',round(pipe.score(Xb,yb),3));print('Candidate holdout:',round(candidate.score(Xb,yb),3))

Baseline holdout: 0.933
Candidate holdout: 0.967


**Interpretation.** One split can suggest which model warrants further investigation. It is not enough to prove that a model change is robust across populations or time.